# Time stepping: pushing at the southern pole

This example uses `hookeanity.System` to generate a moving surface in `snapshots/push`. The system stores internal displacement in a co-rotating body frame. `rotation()` maps that frame into the lab, while `export_vts()` writes the lab-frame shape.

The two panels below show the lab shape with its moving reference axes and the same shape transformed back to the body frame. Colors show total lab displacement on the left and internal displacement on the right. The latter is the interpolated body-frame `sys.surface()`; the inverse transformation uses the reference center returned by `sys.center()`.

Run the notebook with the `hooke` conda environment. The visualization cell reloads orientation metadata saved with the frames.

In [1]:
import math
from pathlib import Path

import ipywidgets as widgets
import numpy as np
import pyvista as pv
from hookeanity import System
from IPython.display import display
from scipy.spatial.transform import Rotation
from tqdm.notebook import tqdm

In [2]:
# Choose parameters and initialize the system

resolution = (10, 100)  # control spatial resolution
lmax = 100  # control resolution of Sobolev-Matérn kernels
dt = 20  # time step
kbt = 4e-4  # effective temperature
gamma = 3e3
f_loc = (math.pi, 0.0)  # point-source force at the southern pole
f_ext = 2e-2  # magnitude of the pushing force
lam, mu = (1.4286, 0.3571)  # Lamé coefficients

method = "strang"  # integration method ("strang", "trotter", "euler")
seed = 0  # specify seed for the random number generator or set it to `None`
# A visible starting orientation makes the moving frame easy to see.
initial_rotation = Rotation.from_euler("xyz", [15, -20, 30], degrees=True).as_matrix()
iterations = 30000  # number of time steps
nframes = 10  # desired number of frames excluding the initial one

each = iterations // nframes
assert each > 1, (
    f"number of iterations {iterations} must be greater than the number of frames {nframes}"
)
sys = System(
    resolution,
    lam,
    mu,
    dt,
    f_ext=f_ext,
    f_loc=f_loc,
    kbt=kbt,
    gamma=gamma,
    rng=seed,
    rotation=initial_rotation,
    method=method,
    lmax=lmax,
    verbose=True,
)

# Prepare the snapshots directory
snapshots = Path("snapshots/push")
snapshots.mkdir(parents=True, exist_ok=True)

# To accelerate snapshot generation
projection = snapshots / "projection.h5"
sys.save_projection_cache(projection, resolution[1])
center = sys.center()  # volume-weighted center of the reference body
steps_per_frame = each
print("Ready to go!")

INFO 2026-09-24 13:48:27 | Initializing system
INFO 2026-09-24 13:48:28 |   derivative operator
INFO 2026-09-24 13:48:42 |   elastic traction operator
INFO 2026-09-24 13:48:42 |   divergence of the elastic stress
INFO 2026-09-24 13:48:42 |   additive forces
INFO 2026-09-24 13:49:35 |   surface reduction diagnostics: rank 2700/2700, cond 1.020e+06, residual 5.024e-10
INFO 2026-09-24 13:49:35 |   Strang traction operator
INFO 2026-09-24 13:49:35 | System ready
Ready to go!


In [3]:
# Run the simulation
snapshot = snapshots / "0.vts"
sys.export_vts(snapshot, resolution[1], projection)
frames = [snapshot]
orientations = [sys.rotation()]
for it in tqdm(range(iterations), desc="Simulation"):
    sys()
    if (it + 1) % each == 0:
        snapshot = snapshots / f"{(it + 1) // each}.vts"
        sys.export_vts(snapshot, resolution[1], projection)
        frames.append(snapshot)
        orientations.append(sys.rotation())

# Finalizing
pvd_content = (
    '<?xml version="1.0"?>\n'
    '<VTKFile type="Collection" version="0.1" byte_order="LittleEndian">\n'
    "  <Collection>\n"
)
for frame in frames:
    timestep = int(frame.stem) * steps_per_frame * dt
    pvd_content += (
        f'    <DataSet timestep="{timestep}" '
        f'group="surface" part="0" file="{frame.name}"/>\n'
    )
pvd_content += "  </Collection>\n</VTKFile>\n"

pvd_file = snapshots / "output.pvd"
pvd_file.write_text(pvd_content)
np.savez_compressed(
    snapshots / "frame_metadata.npz", rotations=np.stack(orientations), center=center
)
print(f"{len(frames)} frames generated")

Simulation:   0%|          | 0/30000 [00:00<?, ?it/s]

11 frames generated


In [ ]:
# Compare lab motion with deformation in the co-rotating frame
pvd_file = snapshots / "output.pvd"
with np.load(snapshots / "frame_metadata.npz") as frame_metadata:
    orientations = frame_metadata["rotations"]
    center = frame_metadata["center"]
reader = pv.get_reader(pvd_file)
times = reader.time_values
assert len(times) == len(orientations)


def get_frame(i):
    reader.set_active_time_point(i)
    mesh = reader.read()
    if isinstance(mesh, pv.MultiBlock) and len(mesh) == 1:
        mesh = mesh[0]
    return mesh


def get_views(i):
    lab = get_frame(i)
    body = lab.copy(deep=True)
    # VTS displacement is measured in the lab from the unrotated reference.
    reference = lab.points - np.asarray(lab.point_data["Displacement_Vector"])
    # Row vectors use R.T from body to lab, and R from lab to body.
    body.points = center + (lab.points - center) @ orientations[i]
    internal = body.points - reference
    body.point_data["Internal_Displacement_Vector"] = internal
    body.point_data["Internal_Displacement_Magnitude"] = np.linalg.norm(
        internal, axis=1
    )
    return lab, body


# Use fixed bounds and color ranges so the frames can be compared directly.
minimum = np.full(3, np.inf)
maximum = np.full(3, -np.inf)
lab_max = internal_max = 0.0
for i in range(len(times)):
    lab, body = get_views(i)
    for mesh in (lab, body):
        minimum = np.minimum(minimum, mesh.points.min(axis=0))
        maximum = np.maximum(maximum, mesh.points.max(axis=0))
    lab_max = max(lab_max, float(np.max(lab.point_data["Displacement_Magnitude"])))
    internal_max = max(
        internal_max, float(np.max(body.point_data["Internal_Displacement_Magnitude"]))
    )
padding = 0.25 * np.maximum(maximum - minimum, 1e-12)
view_bounds = tuple(
    value
    for axis in range(3)
    for value in (minimum[axis] - padding[axis], maximum[axis] + padding[axis])
)
lab_clim = (0, max(lab_max, 1e-12))
internal_clim = (0, max(internal_max, 1e-12))

plotter = pv.Plotter(shape=(1, 2), notebook=True)
actors = {}
axis_colors = ("red", "green", "blue")


def add_frame(i):
    lab, body = get_views(i)
    plotter.subplot(0, 0)
    if "lab" in actors:
        plotter.remove_actor(actors["lab"])
        for actor in actors["axes"]:
            plotter.remove_actor(actor)
    actors["lab"] = plotter.add_mesh(
        lab,
        scalars="Displacement_Magnitude",
        clim=lab_clim,
        name="lab_surface",
        reset_camera=False,
        scalar_bar_args={"title": "Lab displacement"},
    )
    actors["axes"] = [
        plotter.add_mesh(
            pv.Arrow(start=center, direction=orientations[i][:, axis], scale=0.35),
            color=color,
            name=f"body_axis_{axis}",
            reset_camera=False,
        )
        for axis, color in enumerate(axis_colors)
    ]
    angle = np.degrees(
        Rotation.from_matrix(orientations[i] @ orientations[0].T).magnitude()
    )
    plotter.add_text(
        f"Lab frame: t = {times[i]:g}, frame rotation = {angle:.2f}°",
        name="lab_label",
        position="upper_left",
    )

    plotter.subplot(0, 1)
    if "body" in actors:
        plotter.remove_actor(actors["body"])
    actors["body"] = plotter.add_mesh(
        body,
        scalars="Internal_Displacement_Magnitude",
        clim=internal_clim,
        name="body_surface",
        reset_camera=False,
        scalar_bar_args={"title": "Internal displacement"},
    )
    plotter.add_text(
        f"Co-rotating frame: t = {times[i]:g}",
        name="body_label",
        position="upper_left",
    )
    plotter.render()


add_frame(0)
for column in (0, 1):
    plotter.subplot(0, column)
    plotter.add_mesh(
        pv.Box(bounds=view_bounds),
        style="wireframe",
        opacity=0.2,
        color="black",
        name="motion_bounds",
        reset_camera=False,
    )
    plotter.camera_position = "iso"
    plotter.reset_camera(bounds=view_bounds)
plotter.show()

frame_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=len(times) - 1,
    step=1,
    description="Frame",
    continuous_update=False,
)
frame_slider.observe(lambda change: add_frame(change["new"]), names="value")
prev_button = widgets.Button(description="Previous")
next_button = widgets.Button(description="Next")
prev_button.on_click(
    lambda _: setattr(frame_slider, "value", max(0, frame_slider.value - 1))
)
next_button.on_click(
    lambda _: setattr(
        frame_slider, "value", min(len(times) - 1, frame_slider.value + 1)
    )
)
display(widgets.HBox([prev_button, next_button]), frame_slider)

Widget(value='<iframe src="http://localhost:42373/index.html?ui=P_0x7f009c758d70_0&reconnect=auto" class="pyvi…

IntSlider(value=0, continuous_update=False, description='Frame', max=10)